# Holographic Bulk-Boundary Duality & Entanglement Entropy Recipe

This recipe combines 4 `algebrax` tools to model AdS/CFT Holographic Duality and Quantum Entanglement:

1. **Hyperbolic Bulk Geometry** (`algebrax.analysis.forman_ricci_curvature`):
   Evaluates discrete negative Forman-Ricci curvature ($K < 0$) on hyperbolic bulk graphs ($\text{AdS}_3$ space discretization).
2. **Discrete Holographic Gauss-Stokes Divergence** (`algebrax.analysis.divergence`):
   Verifies that net interior bulk field divergence $\sum_{v \in \text{Bulk}} \text{div}(F)_v$ matches boundary edge flux.
3. **MERA Tensor Network Contraction** (`algebrax.trie.AlgebraicTrie`):
   Models a Multiscale Entanglement Renormalization Ansatz (MERA) tree tensor network mapping bulk IR scale nodes to boundary UV sites via subtree contraction (`contract(prefix)`).
4. **Ryu-Takayanagi Entanglement Entropy** (`algebrax.probability.entropy` & `mutual_information`):
   Computes boundary subsystem entanglement entropy $S(A)$ and quantum mutual information $I(A; B)$, matching bulk minimal surface area bounds $S(A) = \frac{\text{Area}(\gamma_A)}{4 G_N}$.

In [ ]:
import algebrax as ax


## 1. Hyperbolic Bulk Geometry (forman_ricci_curvature)

We compute discrete Forman-Ricci curvature on a bulk-boundary graph ($K < 0$ indicates hyperbolic geometry).

In [ ]:

bulk_boundary_graph = {
    0: {1: 1.0, 2: 1.0},
    1: {0: 1.0, 3: 1.0, 4: 1.0},
    2: {0: 1.0, 5: 1.0, 6: 1.0},
    3: {1: 1.0},
    4: {1: 1.0},
    5: {2: 1.0},
    6: {2: 1.0},
}

ricci_k = ax.analysis.forman_ricci_curvature(bulk_boundary_graph)
print('Forman-Ricci Curvature across Graph Edges:')
for edge, k_val in sorted(ricci_k.items()):
    print(f'  Edge {edge}: K = {k_val:+5.2f}')

## 2. Discrete Holographic Gauss-Stokes Divergence (divergence)

We verify the Holographic Gauss-Stokes Theorem: bulk field divergence matches boundary flux.

In [ ]:
import algebrax as ax

edge_flux = {0: {1: 10.0, 2: 14.0}, 1: {3: 4.0, 4: 6.0}, 2: {5: 8.0, 6: 6.0}}
div_f = ax.analysis.divergence(edge_flux)

bulk_div = sum(div_f.get(v, 0.0) for v in [0, 1, 2])
boundary_flux = sum(div_f.get(v, 0.0) for v in [3, 4, 5, 6])

print(f'Total Bulk Divergence: {bulk_div:+6.1f}')
print(f'Total Boundary Flux:   {boundary_flux:+6.1f}')
print(f'Conservation Sum:      {bulk_div + boundary_flux:.1f}')

## 3. MERA Tensor Network Subtree Contraction (AlgebraicTrie)

We construct a MERA tensor network using `AlgebraicTrie` and contract subtree scale branches via `contract(prefix)`.

In [ ]:
import algebrax as ax

mera = ax.trie.AlgebraicTrie()
mera[('IR_Root', 'Scale_1', 'Site_A')] = 0.40
mera[('IR_Root', 'Scale_1', 'Site_B')] = 0.35
mera[('IR_Root', 'Scale_2', 'Site_C')] = 0.15
mera[('IR_Root', 'Scale_2', 'Site_D')] = 0.10

scale_1_weight = mera.contract(('IR_Root', 'Scale_1'))
print('Contracted MERA IR Scale 1 Weight:', scale_1_weight)

## 4. Ryu-Takayanagi Boundary Entanglement Entropy

We calculate boundary subsystem entropy $S(A)$ and quantum mutual information $I(A; B)$.

In [ ]:
import algebrax as ax

boundary_a = {'00': 0.50, '01': 0.25, '10': 0.15, '11': 0.10}
boundary_b = {'00': 0.40, '01': 0.30, '10': 0.20, '11': 0.10}
joint_ab = {
    '00': {'00': 0.30, '01': 0.10},
    '01': {'00': 0.05, '01': 0.20},
    '10': {'10': 0.15, '11': 0.05},
    '11': {'10': 0.05, '11': 0.10},
}

s_a = ax.probability.entropy(boundary_a)
mi = ax.probability.mutual_information(joint_ab)
print(f'Boundary Entanglement Entropy S(A): {s_a:.4f} bits')
print(f'Ryu-Takayanagi Area(gamma_A):        {s_a * 4.0:.4f} (G_N units)')
print(f'Quantum Mutual Information I(A; B):  {mi:.4f} bits')